# VAE1 (Domain A) Training — Kaggle Notebook

Trains VAE1 on real old photos (the GAN+VAE latent-translation method's domain A) using the tested scaffold from the project: `models/blocks.py`, `models/vae.py`, `data/real_photo_dataset.py`, `train_vae_domain_a.py` (the version with mixed precision, `--steps-per-epoch`, live tqdm progress, and a persistent timestamped log file).

**Before running anything:**
1. Right sidebar: **Settings > Accelerator > GPU T4 x2** (or P100)
2. Right sidebar: **Settings > Internet > On** (needed for pip installs; only needed for the data step below if you choose to re-scrape instead of uploading your own data)

**Getting your already-downloaded real photos into this notebook** — you have two options, covered in section 4 below:
- **Recommended**: upload the `real_old_photos` folder you already collected as a Kaggle Dataset from your own machine, then attach it here. This preserves exactly the curated set you already have.
- **Alternative**: re-run `loc_scraper.py` directly in this notebook to build a fresh set in the cloud (useful if you'd rather not upload, but won't be the identical set you already downloaded).

**Persistence**: `/kaggle/working/` is your writable scratch space. Click **Save Version** to commit it as that version's Output. To resume training in a later session, attach your own previous Output as an input dataset (**Add Data > Notebook Output Files**) and copy the latest checkpoint back into `/kaggle/working/` before passing it to `--resume`.


## 1. Check GPU

In [1]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected — go to Settings (right sidebar) > Accelerator > GPU T4 x2.')


CUDA available: True
GPU: Tesla T4


## 2. Install dependencies

In [2]:
!pip install -q pillow tqdm
import torch, torchvision
print('torch', torch.__version__)
print('torchvision', torchvision.__version__)


torch 2.10.0+cu128
torchvision 0.25.0+cu128


## 3. Recreate the project files

Writes out the exact tested files from the project — includes mixed precision, `--steps-per-epoch`, live tqdm progress bar, and the timestamped persistent log file added most recently.


In [3]:
import os
os.chdir('/kaggle/working')
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)


In [4]:
%%writefile models/__init__.py



Writing models/__init__.py


In [5]:
%%writefile models/blocks.py
"""
Basic building blocks shared by the VAE encoder and decoder.

These are standard, well-known layer patterns (residual blocks, strided
conv downsampling, transposed-conv upsampling) -- not specific to any one
paper's architecture. The VAE class that assembles them into the actual
"Bringing Old Photos Back to Life"-style domain VAE is in vae.py, and that
assembly (encoder depth, bottleneck design, how mu/logvar are produced) is
the part you're implementing yourself from the paper's description.
"""

import torch
import torch.nn as nn


class ResidualBlock(nn.Module):
    """A standard two-conv residual block with instance normalization.
    Used inside the encoder/decoder to add capacity without changing
    spatial resolution."""

    def __init__(self, channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, kernel_size=3, padding=0),
            nn.InstanceNorm2d(channels, affine=True),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, kernel_size=3, padding=0),
            nn.InstanceNorm2d(channels, affine=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.block(x)


class DownsampleBlock(nn.Module):
    """Strided conv that halves spatial resolution and doubles channels
    (up to a cap), used to build the encoder's downsampling path."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(out_channels, affine=True),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class UpsampleBlock(nn.Module):
    """Transposed conv that doubles spatial resolution, used to build the
    decoder's upsampling path (mirrors DownsampleBlock)."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(out_channels, affine=True),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


Writing models/blocks.py


In [6]:
%%writefile models/vae.py
"""
A convolutional VAE with a *spatial* latent bottleneck (a small feature map,
not a single flattened vector) rather than the more familiar
flatten-to-a-vector VAE design.

Why spatial: "Bringing Old Photos Back to Life" needs the latent
representation to preserve rough spatial layout, so that later (in the
mapping network you'll build next) a damage mask can be used to tell the
model *where* in the latent space to focus repair. A flattened-vector
latent would throw that spatial correspondence away.

This same class is used for both VAE1 (domain A: real old photos) and VAE2
(domain B: clean photos) -- you'll instantiate two separate copies with
their own weights, one per domain, trained independently in
train_vae_domain_a.py / train_vae_domain_b.py.
"""

import torch
import torch.nn as nn

from models.blocks import ResidualBlock, DownsampleBlock, UpsampleBlock


class Encoder(nn.Module):
    def __init__(self, in_channels=3, base_channels=64, n_downsample=3,
                 n_residual_blocks=4, latent_channels=64, max_channels=512):
        super().__init__()

        # Initial conv, no downsampling yet
        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_channels, base_channels, kernel_size=7, padding=0),
            nn.InstanceNorm2d(base_channels, affine=True),
            nn.ReLU(inplace=True),
        ]

        # Downsampling path: halve spatial resolution each step, double
        # channels up to max_channels
        channels = base_channels
        for _ in range(n_downsample):
            next_channels = min(channels * 2, max_channels)
            layers.append(DownsampleBlock(channels, next_channels))
            channels = next_channels

        # Residual blocks at the bottleneck resolution, adding capacity
        # without further downsampling
        for _ in range(n_residual_blocks):
            layers.append(ResidualBlock(channels))

        self.backbone = nn.Sequential(*layers)

        # Separate 1x1 convs producing the mean and log-variance maps of
        # the latent distribution -- same spatial size as the backbone
        # output, just a different channel count
        self.to_mu = nn.Conv2d(channels, latent_channels, kernel_size=1)
        self.to_logvar = nn.Conv2d(channels, latent_channels, kernel_size=1)

        self.bottleneck_channels = channels

    def forward(self, x: torch.Tensor):
        features = self.backbone(x)
        mu = self.to_mu(features)
        logvar = self.to_logvar(features)
        logvar = torch.clamp(logvar, min=-10.0, max=10.0)
        return mu, logvar


class Decoder(nn.Module):
    def __init__(self, out_channels=3, base_channels=64, n_downsample=3,
                 n_residual_blocks=4, latent_channels=64, max_channels=512):
        super().__init__()

        # Figure out the bottleneck channel count the same way the
        # encoder did, so the shapes line up
        channels = base_channels
        for _ in range(n_downsample):
            channels = min(channels * 2, max_channels)

        layers = [nn.Conv2d(latent_channels, channels, kernel_size=1)]

        for _ in range(n_residual_blocks):
            layers.append(ResidualBlock(channels))

        # Upsampling path: mirror of the encoder's downsampling path
        channel_sequence = []
        c = base_channels
        for _ in range(n_downsample):
            channel_sequence.append(min(c * 2, max_channels))
            c = min(c * 2, max_channels)
        channel_sequence = [base_channels] + channel_sequence
        # channel_sequence e.g. [64, 128, 256, 512] for n_downsample=3;
        # we walk it backwards to go from bottleneck back to base_channels
        for i in range(n_downsample):
            in_ch = channel_sequence[n_downsample - i]
            out_ch = channel_sequence[n_downsample - i - 1]
            layers.append(UpsampleBlock(in_ch, out_ch))

        layers += [
            nn.ReflectionPad2d(3),
            nn.Conv2d(base_channels, out_channels, kernel_size=7, padding=0),
            nn.Tanh(),  # output in [-1, 1], matches how we'll normalize images
        ]

        self.net = nn.Sequential(*layers)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


class DomainVAE(nn.Module):
    """
    Full VAE: encode -> reparameterize -> decode.

    Instantiate one of these per domain (real old photos / clean photos).
    The `Encoder`/`Decoder` above are shared *class* definitions but each
    DomainVAE instance gets its own independently-trained weights.
    """

    def __init__(self, in_channels=3, base_channels=64, n_downsample=3,
                 n_residual_blocks=4, latent_channels=64, max_channels=512):
        super().__init__()
        self.encoder = Encoder(in_channels, base_channels, n_downsample,
                                n_residual_blocks, latent_channels, max_channels)
        self.decoder = Decoder(in_channels, base_channels, n_downsample,
                                n_residual_blocks, latent_channels, max_channels)

    @staticmethod
    def reparameterize(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """The standard VAE reparameterization trick: sample z = mu + eps*std
        where eps ~ N(0, 1), so gradients can flow through the sampling step."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x: torch.Tensor):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar


def vae_loss(recon: torch.Tensor, target: torch.Tensor, mu: torch.Tensor,
             logvar: torch.Tensor, kl_weight: float = 1.0):
    """
    Standard VAE loss = reconstruction term + KL divergence term.

    Reconstruction uses L1 (tends to give sharper results than MSE for
    images -- this is a common choice in image-translation VAEs, not
    something unique to this paper).

    KL divergence pulls the latent distribution toward a standard normal,
    which is what makes the latent space smooth/well-structured enough for
    the mapping network to later translate between domains.
    """
    recon_loss = torch.nn.functional.l1_loss(recon, target)

    # Closed-form KL divergence between N(mu, sigma^2) and N(0, 1),
    # averaged over batch and spatial/channel dims
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    total_loss = recon_loss + kl_weight * kl_loss
    return total_loss, recon_loss, kl_loss


Writing models/vae.py


In [7]:
%%writefile data/__init__.py



Writing data/__init__.py


In [8]:
%%writefile data/real_photo_dataset.py
"""
Dataset loader for VAE1 (domain A) training: real old photos, as collected
by loc_scraper.py / dpla_scraper.py. Unsupervised -- no labels needed, just
a folder of images.
"""

import os
import csv
import random

from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T


class RealOldPhotoDataset(Dataset):
    """
    Expects the folder layout produced by loc_scraper.py / dpla_scraper.py:

        <root>/images/*.jpg
        <root>/manifest.csv   (optional, only used to list valid filenames)

    If manifest.csv is present, filenames are read from it (keeps you in
    sync with whatever passed your download-time validation). Otherwise
    falls back to globbing every image file in <root>/images/.
    """

    def __init__(self, root: str, image_size: int = 256, augment: bool = True):
        self.root = root
        self.images_dir = os.path.join(root, "images")
        self.image_size = image_size
        self.augment = augment

        self.filenames = self._load_filenames()
        if len(self.filenames) == 0:
            raise ValueError(f"No images found under {self.images_dir}")

        # Resize the short side up a bit past image_size so RandomCrop has
        # room to move -- this is a standard cheap augmentation for
        # unsupervised reconstruction training.
        load_size = int(image_size * 1.12)

        transform_list = [
            T.Resize(load_size),
        ]
        if augment:
            transform_list += [
                T.RandomCrop(image_size),
                T.RandomHorizontalFlip(),
            ]
        else:
            transform_list += [T.CenterCrop(image_size)]

        transform_list += [
            T.ToTensor(),
            T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  # -> [-1, 1]
        ]
        self.transform = T.Compose(transform_list)

    def _load_filenames(self):
        manifest_path = os.path.join(self.root, "manifest.csv")
        if os.path.exists(manifest_path):
            filenames = []
            with open(manifest_path, "r", encoding="utf-8") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    fn = row.get("local_filename")
                    if fn:
                        filenames.append(fn)
            if filenames:
                return filenames

        # fallback: glob the images directory directly
        if not os.path.isdir(self.images_dir):
            return []
        valid_ext = (".jpg", ".jpeg", ".png")
        return [f for f in os.listdir(self.images_dir) if f.lower().endswith(valid_ext)]

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        path = os.path.join(self.images_dir, filename)
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            # If something slipped past validation and is unreadable,
            # fall back to a random other item rather than crashing an
            # entire training run over one bad file.
            return self.__getitem__(random.randrange(len(self)))

        img_tensor = self.transform(img)
        return {"image": img_tensor, "filename": filename}


def denormalize(tensor: torch.Tensor) -> torch.Tensor:
    """Inverse of the Normalize(mean=0.5, std=0.5) above, for saving/viewing
    reconstructed images. Maps [-1, 1] back to [0, 1]."""
    return (tensor * 0.5 + 0.5).clamp(0, 1)


Writing data/real_photo_dataset.py


In [9]:
%%writefile train_vae_domain_a.py
"""
Train VAE1 (domain A) on real old photos, unsupervised reconstruction.

This is the first, smallest independently-testable piece of the full
pipeline: encoder -> reparameterize -> decoder, trained to reconstruct
real old photos. Once this trains stably, VAE2 (domain B, on clean photos)
uses the exact same architecture/training loop against a different
dataset -- and both feed into the mapping network stage after that.

Usage:
    python train_vae_domain_a.py --data-root ./real_old_photos --epochs 50 \
        --batch-size 8 --image-size 256 --out-dir ./runs/vae_domain_a
"""

import argparse
import os
import time
from datetime import datetime

import torch
from torch.utils.data import DataLoader, RandomSampler
import torchvision.utils as vutils
from tqdm import tqdm

from models.vae import DomainVAE, vae_loss
from data.real_photo_dataset import RealOldPhotoDataset, denormalize


def format_duration(seconds):
    seconds = int(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours:
        return f"{hours}h {minutes}m {secs}s"
    if minutes:
        return f"{minutes}m {secs}s"
    return f"{secs}s"


class Logger:
    """Prints with a wall-clock timestamp AND appends the same line to a log
    file, so you can check progress from outside the live session (e.g.
    tailing the file over SSH, or reopening a Kaggle notebook that's still
    running) rather than only trusting that the visible cell output is
    current. Opens in append mode, so resuming a run continues the same log
    rather than overwriting history."""

    def __init__(self, log_path):
        self.log_path = log_path
        os.makedirs(os.path.dirname(log_path) or ".", exist_ok=True)
        self._file = open(log_path, "a", encoding="utf-8")

    def log(self, msg):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{timestamp}] {msg}"
        print(line, flush=True)
        self._file.write(line + "\n")
        self._file.flush()

    def close(self):
        self._file.close()


def save_reconstruction_grid(model, batch, out_path, device, max_images=8):
    """Saves a side-by-side grid of [original | reconstruction] so you can
    visually sanity-check training progress, not just watch the loss curve."""
    model.eval()
    with torch.no_grad():
        images = batch["image"][:max_images].to(device)
        recon, _, _ = model(images)
        comparison = torch.cat([denormalize(images), denormalize(recon)], dim=0)
        vutils.save_image(comparison, out_path, nrow=max_images)
    model.train()


def main():
    parser = argparse.ArgumentParser(description="Train VAE1 on real old photos (domain A).")
    parser.add_argument("--data-root", type=str, required=True,
                         help="folder containing images/ and manifest.csv from the scraper scripts")
    parser.add_argument("--out-dir", type=str, default="./runs/vae_domain_a")
    parser.add_argument("--image-size", type=int, default=256)
    parser.add_argument("--batch-size", type=int, default=8,
                         help="the official repo uses 80-120 across 4 GPUs; start much smaller on one GPU/CPU")
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--kl-weight", type=float, default=0.01,
                         help="weight on the KL term; too high early on can collapse reconstructions to blur")
    parser.add_argument("--latent-channels", type=int, default=128)
    parser.add_argument("--n-downsample", type=int, default=3)
    parser.add_argument("--n-residual-blocks", type=int, default=4)
    parser.add_argument("--num-workers", type=int, default=4)
    parser.add_argument("--save-every", type=int, default=5, help="save a checkpoint every N epochs")
    parser.add_argument("--sample-every", type=int, default=200,
                         help="save a reconstruction sample grid every N training steps")
    parser.add_argument("--resume", type=str, default=None, help="path to a checkpoint to resume from")
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--amp", action="store_true",
                         help="use automatic mixed precision (fp16) training -- meaningful speedup on modern "
                              "GPUs (T4 and newer) with minimal code cost; no effect on CPU")
    parser.add_argument("--steps-per-epoch", type=int, default=None,
                         help="if set, each 'epoch' samples this many random batches (with replacement) instead "
                              "of iterating the full dataset once -- use this to shorten epoch wall-clock time.")
    parser.add_argument("--log-every", type=int, default=20,
                         help="print a data-loading-vs-compute timing breakdown every N steps. Set to 0 to disable.")
    args = parser.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    checkpoints_dir = os.path.join(args.out_dir, "checkpoints")
    samples_dir = os.path.join(args.out_dir, "samples")
    os.makedirs(checkpoints_dir, exist_ok=True)
    os.makedirs(samples_dir, exist_ok=True)

    logger = Logger(os.path.join(args.out_dir, "train_log.txt"))
    log = logger.log

    device = torch.device(args.device)
    log(f"Using device: {device}")
    if device.type == "cuda":
        log(f"  GPU: {torch.cuda.get_device_name(device)}")
    cpu_count = os.cpu_count()
    log(f"  CPUs available: {cpu_count}, --num-workers set to {args.num_workers}")
    if args.num_workers > cpu_count:
        log(f"  Warning: --num-workers ({args.num_workers}) exceeds available CPUs ({cpu_count}); "
            f"this can hurt rather than help. Consider lowering it.")

    log("Building dataset index (scanning images/ and manifest.csv)...")
    dataset = RealOldPhotoDataset(args.data_root, image_size=args.image_size, augment=True)
    log(f"Loaded {len(dataset)} real old photos from {args.data_root}")

    if args.steps_per_epoch:
        num_samples = args.steps_per_epoch * args.batch_size
        sampler = RandomSampler(dataset, replacement=True, num_samples=num_samples)
        dataloader = DataLoader(dataset, batch_size=args.batch_size, sampler=sampler,
                                 num_workers=args.num_workers, drop_last=True, pin_memory=(device.type == "cuda"),
                                 persistent_workers=(args.num_workers > 0))
        log(f"Using --steps-per-epoch {args.steps_per_epoch}: each epoch samples "
            f"{num_samples} images (with replacement) instead of the full {len(dataset)}-image dataset")
    else:
        dataloader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True,
                                 num_workers=args.num_workers, drop_last=True, pin_memory=(device.type == "cuda"),
                                 persistent_workers=(args.num_workers > 0))

    # A small fixed batch, held out purely for visualizing reconstruction
    # quality over time (not used for any gradient updates). This first
    # dataloader fetch is also what spins up worker processes -- can take a
    # few seconds even before any training happens, hence the explicit log.
    log("Fetching a fixed sample batch for visualization (this also starts the dataloader workers)...")
    fixed_batch = next(iter(dataloader))
    log("Dataset ready.")

    model = DomainVAE(
        in_channels=3,
        n_downsample=args.n_downsample,
        n_residual_blocks=args.n_residual_blocks,
        latent_channels=args.latent_channels,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, betas=(0.5, 0.999))

    use_amp = args.amp and device.type == "cuda"
    if args.amp and device.type != "cuda":
        log("Note: --amp has no effect on CPU, ignoring.")
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

    start_epoch = 1
    global_step = 0
    if args.resume:
        log(f"Resuming from {args.resume}")
        ckpt = torch.load(args.resume, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if "scaler_state_dict" in ckpt:
            scaler.load_state_dict(ckpt["scaler_state_dict"])
        start_epoch = ckpt["epoch"] + 1
        global_step = ckpt.get("global_step", 0)

    num_params = sum(p.numel() for p in model.parameters())
    log(f"Model has {num_params:,} parameters")
    log(f"Starting training: epochs {start_epoch}-{args.epochs}")

    start_time = time.time()

    for epoch in range(start_epoch, args.epochs + 1):
        epoch_start = time.time()
        running_loss, running_recon, running_kl = 0.0, 0.0, 0.0
        running_data_time, running_compute_time = 0.0, 0.0

        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch}/{args.epochs}", unit="batch", leave=False)

        batch_end_time = time.time()
        for batch in progress_bar:
            data_time = time.time() - batch_end_time

            compute_start = time.time()
            images = batch["image"].to(device, non_blocking=True)

            optimizer.zero_grad()

            with torch.amp.autocast(device.type, enabled=use_amp):
                recon, mu, logvar = model(images)
                loss, recon_loss, kl_loss = vae_loss(recon, images, mu, logvar, kl_weight=args.kl_weight)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            if device.type == "cuda":
                torch.cuda.synchronize()
            compute_time = time.time() - compute_start

            running_loss += loss.item()
            running_recon += recon_loss.item()
            running_kl += kl_loss.item()
            running_data_time += data_time
            running_compute_time += compute_time
            global_step += 1

            # Live, continuously-updating feedback -- this is what tells you
            # "still working" second by second, rather than waiting on a
            # periodic print that might be minutes away.
            progress_bar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "data_t": f"{data_time:.2f}s",
                "compute_t": f"{compute_time:.2f}s",
            })

            if args.log_every and global_step % args.log_every == 0:
                msg = (f"  step {global_step}: data_time={data_time:.3f}s compute_time={compute_time:.3f}s "
                       f"({'data-loading-bound' if data_time > compute_time else 'compute-bound'})")
                if device.type == "cuda":
                    mem_alloc = torch.cuda.memory_allocated(device) / 1e9
                    mem_reserved = torch.cuda.memory_reserved(device) / 1e9
                    msg += f" | GPU mem: {mem_alloc:.2f}GB alloc / {mem_reserved:.2f}GB reserved"
                log(msg)

            if global_step % args.sample_every == 0:
                sample_path = os.path.join(samples_dir, f"step_{global_step:07d}.png")
                save_reconstruction_grid(model, fixed_batch, sample_path, device)
                log(f"  Saved sample grid: {sample_path}")

            batch_end_time = time.time()

        n_batches = len(dataloader)
        elapsed = time.time() - start_time
        log(f"[Epoch {epoch}/{args.epochs}] "
            f"loss={running_loss / n_batches:.4f} "
            f"recon={running_recon / n_batches:.4f} "
            f"kl={running_kl / n_batches:.4f} "
            f"avg_data_time={running_data_time / n_batches:.3f}s "
            f"avg_compute_time={running_compute_time / n_batches:.3f}s "
            f"epoch_time={format_duration(time.time() - epoch_start)} "
            f"total_elapsed={format_duration(elapsed)}")

        if epoch % args.save_every == 0 or epoch == args.epochs:
            ckpt_path = os.path.join(checkpoints_dir, f"vae_domain_a_epoch{epoch:04d}.pt")
            torch.save({
                "epoch": epoch,
                "global_step": global_step,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "args": vars(args),
            }, ckpt_path)
            log(f"  Saved checkpoint: {ckpt_path}")

    total_elapsed = time.time() - start_time
    log(f"Training complete. Total time: {format_duration(total_elapsed)}")
    logger.close()


if __name__ == "__main__":
    main()


Writing train_vae_domain_a.py


## 4a. RECOMMENDED: attach your already-downloaded photos as a Kaggle Dataset

Run this from your **own machine** (where you already have the `real_old_photos` folder and Python installed from running the scraper):

```bash
pip install kaggle
# Get an API token: kaggle.com > your profile > Settings > API > Create New Token
# Place kaggle.json at ~/.kaggle/kaggle.json

cd real_old_photos
kaggle datasets init -p .
# Edit the generated dataset-metadata.json: set a title, e.g. "real-old-photos-va e1"
kaggle datasets create -p .
```

Then in this notebook: right sidebar **Add Data > Your Datasets** > select the dataset you just created > Add. It appears read-only at `/kaggle/input/<your-dataset-slug>/`.

Set `DATA_ROOT` below to that path once attached, then skip section 4b.

In [10]:
# After attaching your dataset via Add Data, point DATA_ROOT at it:
DATA_ROOT = '/kaggle/input/datasets/dorast/real-old-photos-vae1'  # <- adjust to your actual dataset slug

import os
if os.path.isdir(DATA_ROOT):
    n_images = len([f for f in os.listdir(os.path.join(DATA_ROOT, 'images'))
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f'Found {n_images} images at {DATA_ROOT}')
else:
    print(f'{DATA_ROOT} not found yet -- attach your dataset via Add Data first, '
          f'or use section 4b to scrape fresh instead.')


Found 2978 images at /kaggle/input/datasets/dorast/real-old-photos-vae1


## 4b. ALTERNATIVE: scrape fresh real old photos directly in this notebook

Only run this if you're not uploading your existing data. Recreates `loc_scraper.py` and runs it here — requires **Internet: On** in Settings.

In [11]:
# Example: weighted multi-category scrape, matching the balancing approach
# from earlier in the project. Adjust categories/weights/total as needed.
# DATA_ROOT = '/kaggle/working/real_old_photos'
# !python loc_scraper.py \
#     --categories "portrait photograph,street scene,farm landscape,railroad,parade" \
#     --weights "0.15,0.25,0.2,0.2,0.2" \
#     --total-images 3000 \
#     --out-dir "$DATA_ROOT"


## 5. Quick smoke test (small subset, low resolution)

Confirms the pipeline runs end to end on this Kaggle GPU before committing to a long run. A few epochs on a small crop of your data — reconstructions won't look good yet, that's expected and not the point of this test.

In [12]:
#!python train_vae_domain_a.py \
  #  --data-root "$DATA_ROOT" \
   # --epochs 3 \
  #  --batch-size 8 \
  #  --image-size 128 \
  #  --num-workers 2 \
  #  --steps-per-epoch 10 \
  #  --log-every 5 \
  #  --sample-every 10 \
  #  --save-every 1 \
  #  --amp \
  #  --out-dir ./runs/vae_domain_a_smoke_test \
 #   --device cuda


## 6. View a reconstruction sample

Top row = real old photos, bottom row = VAE reconstructions.

In [13]:
#import glob
#from PIL import Image
#import matplotlib.pyplot as plt

#sample_files = sorted(glob.glob('runs/vae_domain_a_smoke_test/samples/*.png'))
#if sample_files:
 #   img = Image.open(sample_files[-1])
  #  plt.figure(figsize=(16, 5))
   # plt.imshow(img)
    #plt.axis('off')
   # plt.title(f'Latest sample: {sample_files[-1]}')
   # plt.show()
#else:
 #   print('No samples found yet — check the training cell above ran successfully.')


## 7. Baseline sanity check: short run at full resolution

8 epochs at your real target resolution/batch size on your full dataset, before committing to a long unattended run. Writes to a separate folder so it won't clash with a real full run.

In [14]:
#!python train_vae_domain_a.py \
 #   --data-root "$DATA_ROOT" \
  #  --epochs 8 \
   # --batch-size 16 \
    #--image-size 128 \
    #--num-workers 2 \
    #--log-every 20 \
    #--sample-every 50 \
    #--save-every 4 \
   # --amp \
    #--out-dir ./runs/baseline_check \
    #--device cuda

# Check the printed epoch_time to estimate full-run duration.


In [15]:
#import glob
#from PIL import Image
#import matplotlib.pyplot as plt

#sample_files = sorted(glob.glob('runs/baseline_check/samples/*.png'))
#if sample_files:
 #   img = Image.open(sample_files[-1])
  #  plt.figure(figsize=(16, 5))
   # plt.imshow(img)
   # plt.axis('off')
   # plt.title(f'Latest sample: {sample_files[-1]}')
   # plt.show()
#else:
 #   print('No samples found yet — check the training cell above ran successfully.')

## 8. Full training run

Once the baseline check looks right, scale up. Remember to **Save Version** before a Kaggle session ends so `/kaggle/working/runs/...` is preserved. Check `train_log.txt` inside your run's output folder for a full timestamped history if you ever need to check progress after the fact.


In [16]:

 # !python train_vae_domain_a.py \
 #     --data-root "$DATA_ROOT" \
 #     --epochs 80 \
 #   --batch-size 16 \
 #    --image-size 128 \
 #     --num-workers 2 \
 #     --latent-channels 128 \
 #   --amp \
 #    --out-dir ./runs/vae_domain_a_v2 \
 #    --device cuda


In [17]:
import subprocess

result = subprocess.run([
    'python', 'train_vae_domain_a.py',
    '--data-root', DATA_ROOT,
    '--epochs', '300',
    '--batch-size', '16',
    '--image-size', '128',
    '--num-workers', '2',
    '--save-every', '20',
    '--latent-channels', '128',
    '--amp',
    '--out-dir', './runs/vae_domain_a_v2',
    '--device', 'cuda',
    '--resume', '/kaggle/input/notebooks/dorast/vae-a/runs/vae_domain_a_v2/checkpoints/vae_domain_a_epoch0160.pt',
])
result.check_returncode()

[2026-09-13 13:58:32] Using device: cuda
[2026-09-13 13:58:32]   GPU: Tesla T4
[2026-09-13 13:58:32]   CPUs available: 4, --num-workers set to 2
[2026-09-13 13:58:32] Building dataset index (scanning images/ and manifest.csv)...
[2026-09-13 13:58:32] Loaded 2978 real old photos from /kaggle/input/datasets/dorast/real-old-photos-vae1
[2026-09-13 13:58:32] Fetching a fixed sample batch for visualization (this also starts the dataloader workers)...
[2026-09-13 13:58:33] Dataset ready.
[2026-09-13 13:58:33] Resuming from /kaggle/input/notebooks/dorast/vae-a/runs/vae_domain_a_v2/checkpoints/vae_domain_a_epoch0160.pt
[2026-09-13 13:58:36] Model has 42,294,531 parameters
[2026-09-13 13:58:36] Starting training: epochs 161-300


Epoch 161/300:  11%|█         | 20/186 [00:04<00:21,  7.68batch/s, loss=0.0635, data_t=0.00s, compute_t=0.13s]

[2026-09-13 13:58:41]   step 29780: data_time=0.000s compute_time=0.129s (compute-bound) | GPU mem: 0.86GB alloc / 1.56GB reserved


Epoch 161/300:  21%|██        | 39/186 [00:07<00:19,  7.72batch/s, loss=0.0747, data_t=0.00s, compute_t=0.13s]

[2026-09-13 13:58:43]   step 29800: data_time=0.000s compute_time=0.130s (compute-bound) | GPU mem: 0.86GB alloc / 1.56GB reserved
[2026-09-13 13:58:43]   Saved sample grid: ./runs/vae_domain_a_v2/samples/step_0029800.png


...

Epoch 300/300:  89%|████████▉ | 166/186 [00:23<00:02,  7.07batch/s, loss=0.0511, data_t=0.00s, compute_t=0.14s]

[2026-09-13 15:00:31]   step 55780: data_time=0.000s compute_time=0.140s (compute-bound) | GPU mem: 0.86GB alloc / 1.86GB reserved


Epoch 300/300:  99%|█████████▉| 185/186 [00:26<00:00,  7.10batch/s, loss=0.0568, data_t=0.00s, compute_t=0.14s]

[2026-09-13 15:00:34]   step 55800: data_time=0.000s compute_time=0.140s (compute-bound) | GPU mem: 0.86GB alloc / 1.86GB reserved
[2026-09-13 15:00:34]   Saved sample grid: ./runs/vae_domain_a_v2/samples/step_0055800.png
[2026-09-13 15:00:34] [Epoch 300/300] loss=0.0579 recon=0.0526 kl=0.5264 avg_data_time=0.001s avg_compute_time=0.141s epoch_time=26s total_elapsed=1h 1m 58s


[2026-09-13 15:00:35]   Saved checkpoint: ./runs/vae_domain_a_v2/checkpoints/vae_domain_a_epoch0300.pt
[2026-09-13 15:00:35] Training complete. Total time: 1h 1m 58s


In [18]:
# Example full run:
# !python train_vae_domain_a.py \
#     --data-root "$DATA_ROOT" \
#     --epochs 50 \
#     --batch-size 16 \
#     --image-size 256 \
#     --num-workers 2 \
#     --amp \
#     --out-dir ./runs/vae_domain_a \
#     --device cuda

# To resume from a previous session's checkpoint (attach its Output as an
# input dataset first, then point --resume at the copied-in checkpoint):
# !python train_vae_domain_a.py \
#     --data-root "$DATA_ROOT" \
#     --epochs 50 \
#     --batch-size 16 \
#     --image-size 256 \
#     --amp \
#     --out-dir ./runs/vae_domain_a \
#     --device cuda \
#     --resume /kaggle/working/runs/vae_domain_a/checkpoints/<latest>.pt


In [19]:
#!python train_vae_domain_a.py \
 #    --data-root "$DATA_ROOT" \
  # --epochs 50 \
   # --batch-size 16 \
    # --image-size 256 \
    # --num-workers 2 \
   # --amp \
    #--out-dir ./runs/vae_domain_a \
   # --device cuda

In [20]:
#import glob
#ckpts = sorted(glob.glob('runs/vae_domain_a/checkpoints/*.pt'))
#print(ckpts)

In [21]:
import shutil

for folder in ['/kaggle/working/real_photos_val']:
    if os.path.isdir(folder):
        shutil.rmtree(folder)
        print(f'Deleted: {folder}')
    else:
        print(f'Not found (already clean): {folder}')

Not found (already clean): /kaggle/working/real_photos_val
